In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")


✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_pythWL,dog_pythWL,fav_Luck,dog_Luck,fav_last10,dog_last10,fav_last20,dog_last20,fav_last30,dog_last30,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-06T13:10:00-04:00,TB,None,MIA,None,MIA,None,TB,None,-1.5,1.5,0.420,0.597,138,-148,0,TB -1.5,Zack Littell,Edward Cabrera,3.86,4.14,1.10,1.42,48.0,48.0,74.2,45.2,0.254,0.261,0.532,0.383,.246,.247,3.43,5.04,33,23,29,37,W 3,L 4,4.4,4.0,3.6,5.4,0.0,0.1,0.8,-1.3,36-26,22-38,-3.0,1.0,7-3,3-7,14-6,8-12,19-11,11-19,4.35,4.00,2291,2268,2067,2039,270,240,508,503,96,100,3,8,82,72,257,230,86,48,187,186,525,492,.246,.247,.311,.313,.388,.379,.699,.691,100,90,3.60,5.38,3.43,5.04,15,11,477,542,223,323,211,299,164,221,488,476,114,87,4.25,4.41,1.156,1.429,7.7,9.1
1,2025-06-06T19:40:00-04:00,KAN,None,CWS,None,CWS,None,KAN,None,-1.5,1.5,0.452,0.561,121,-128,0,CWS +1.5,Seth Lugo,Davis Martin,3.45,3.67,1.15,1.24,45.0,41.0,60.0,68.2,0.237,0.260,0.524,0.317,.250,.223,3.23,4.16,33,20,30,43,W 1,W 1,3.4,3.4,3.5,4.3,0.1,0.0,0.1,-0.9,31-32,25-38,2.0,-5.0,5-5,3-7,8-12,6-14,16-14,11-19,3.43,3.44,2327,2301,2120,2052,216,217,531,458,106,94,12,3,60,67,213,210,51,47,149,206,434,525,.250,.223,.303,.296,.367,.342,.670,.638,88,82,3.48,4.35,3.23,4.16,21,5,492,512,219,274,202,251,178,233,512,458,128,99,3.69,4.50,1.191,1.373,7.9,8.5
2,2025-06-06T20:40:00-04:00,NYM,None,COL,None,COL,None,NYM,None,-1.5,1.5,0.646,0.374,-182,167,0,COL +1.5,Kodai Senga,Antonio Senzatela,1.60,7.14,1.18,1.98,59.0,31.0,62.0,58.0,0.203,0.377,0.619,0.194,.244,.216,2.85,5.39,39,12,24,50,L 1,W 3,4.4,3.2,3.3,6.1,-0.1,0.3,1.1,-2.6,40-23,14-48,-1.0,-2.0,7-3,3-7,11-9,5-15,17-13,6-24,4.44,3.16,2389,2250,2090,2047,280,196,511,443,105,94,13,16,44,75,271,194,49,30,230,163,483,599,.244,.216,.328,.279,.413,.356,.741,.635,113,70,3.32,6.08,2.85,5.39,19,10,465,625,209,377,178,322,226,210,565,406,134,86,3.42,4.65,1.231,1.554,7.5,10.5
3,2025-06-06T22:15:00-04:00,ATL,None,SF,None,SF,None,ATL,None,-1.5,1.5,0.432,0.600,131,-150,0,SF +1.5,Spencer Schwellenbach,Hayden Birdsong,3.13,2.37,1.03,1.24,71.0,40.0,74.2,38.0,0.229,0.243,0.443,0.556,.245,.231,3.82,3.04,27,35,34,28,L 4,W 2,4.1,4.1,4.0,3.4,0.0,-0.1,0.1,0.6,31-30,37-26,-4.0,-2.0,2-8,4-6,7-13,10-10,13-17,15-15,4.10,4.11,2313,2339,2069,2074,250,259,506,479,87,94,6,9,70,42,240,248,36,39,209,219,523,532,.245,.231,.318,.309,.388,.370,.706,.679,98,96,4.00,3.44,3.82,3.04,10,19,473,479,244,217,229,188,205,187,533,547,107,126,4.07,3.21,1.258,1.198,7.9,7.8
4,2025-06-06T21:38:00-04:00,SEA,None,LAA,None,LAA,None,SEA,None,-1.5,1.5,0.441,0.574,127,-135,0,LAA +1.5,Bryce Miller,Kyle Hendricks,5.36,5.34,1.53,1.29,37.0,39.0,43.2,59.0,0.271,0.256,0.525,0.459,.236,.226,3.87,4.88,32,28,29,33,L 3,L 1,4.4,4.2,4.4,5.1,-0.2,0.0,-0.2,-0.9,31-30,25-36,1.0,3.0,3-7,3-7,9-11,11-9,13-17,16-14,4.41,4.18,2345,2235,2070,2030,269,255,488,459,74,86,3,7,66,80,260,246,56,25,222,157,551,591,.236,.226,.318,.288,.392,.405,.709,.693,109,95,4.38,5.11,3.87,4.88,18,15,546,559,267,312,239,290,188,256,489,451,96,85,4.00,4.89,1.321,1.522,8.8,9.4
5,2025-06-06T19:05:00-04:00,NYY,None,BOS,None,BOS,None,NYY,None,-1.5,1.5,0.476

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-06.csv
